<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW2a_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW2a: Separating the Residual from the Regional

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Due:** Sunday, October 4, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) via Brightspace


## What you will do

HW1 had one day of readings, one [base station](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#base-station) read twice, and one correction. This survey is a line of twenty stations across a known karst sinkhole at the University of South Florida GeoPark, read over three field days, with the base station re-occupied between every science station. You will:

1. Load the survey and work out what the raw gravity column can and cannot show.
2. Fit the [drift](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#drift) on each day as a [least-squares](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#least-squares) line through the base reads, with an [error bar](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#error-bar-uncertainty) on the slope, and tie the three days to a common reference.
3. Apply the free-air correction and the Bouguer slab correction at the textbook density of 2.67 g/cm³ and locate the lowest point on the reduced profile.
4. Fit a straight-line regional to the reduced profile and remove it, leaving the [residual](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual).

This is the reduction. HW2b takes the residual you produce here and asks what it means.

## What you will hand in

This same notebook with the plots run and the short answers filled in. Save it as `HW2a_LASTNAME.ipynb` and upload to Brightspace.

## Data source and citation

> Parsekian, A. (n.d.). *IGUaNA Unit 3: Gravity and Magnetics Field Data Exercises, Part 1 (USF GeoPark sinkhole survey).* Science Education Resource Center, Carleton College. CC-BY-NC-SA 4.0. [SERC: IGUaNA Unit 3 teaching materials](https://serc.carleton.edu/iguana/teaching_materials/grav_mag/unit3.html)

The survey is a straight line across a campus field in Tampa, Florida, over karst limestone under a cover of sand and soil. The instrument was a relative gravimeter; readings are in milligals (mGal). The base station is at along-profile coordinate 100 m and was re-occupied between every science station. The survey ran in three field sessions:

| Date | Local time | Science stations read (along-profile m) |
|---|---|---|
| 2019-04-04 | 18:29 to 19:25 | 110, 120, 130, 140, 150 |
| 2019-04-06 | 10:27 to 13:18 | 50, 60, 70, 80, 90, 160, 170, 180, 190, 200 |
| 2019-04-18 | 15:52 to 17:12 | 0, 10, 20, 30, 40 |

The along-profile coordinate increases toward the south, so stations 0 to 90 are north of the base and stations 110 to 200 are south of it.

## Loading the data

The code below loads `profile.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `profile.csv` from the Brightspace HW2a page.
2. In Google Colab, click the **📁 Files** icon in the left sidebar.
3. Drag `profile.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("profile.csv")` line instead.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports give us NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

## Part 1: Load and inspect

The file has 43 rows: 23 base-station re-occupations, tagged `base100*` with `is_base` True, and 20 science stations. The columns the reduction uses:

| Column | Meaning |
|---|---|
| `point_along_profile_m` | position along the line; the base is at 100 m |
| `elev_rel_base_m` | station elevation relative to the base, in metres, positive above it |
| `time_est` and `time_since_beg_min` | clock time, and minutes since the first reading of the survey |
| `gravity_mgal` | the [gravimeter](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#gravimeter) reading, in [mGal](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#milligal-mgal) |
| `sd_mgal` | the instrument's own uncertainty on that one reading |
| `is_base` | True for a base re-occupation |

The four position columns (`northing_m`, `easting_m`, `lat_deg`, `lon_deg`) place the stations on the ground in UTM Zone 17N and WGS-84. The reduction below does not read them.

The cell below reads `profile.csv` from the course repository with `pd.read_csv` into a table named `df`, one row per reading. `pd.to_datetime` converts the `time_est` column from text to clock times, and a new column `date` holds the calendar day of each reading so that later cells can split the survey by field day. `df.head(10)` displays the first ten rows.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/profile.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download profile.csv from the
# Brightspace HW2 page, drag it into the Colab file panel, then comment out the
# two lines above and use:
# df = pd.read_csv("profile.csv")

df['time_est'] = pd.to_datetime(df['time_est'])
df['date'] = df['time_est'].dt.date
df.head(10)

This cell reads `df` and counts its rows, its base reads and its science reads from the `is_base` column. It then loops over the three values of `date` and, for each day, prints the number of base reads and the sorted list of science stations read that day. The printout is a check against the field-session table in the data source section; the cell stores nothing.

In [ ]:
print(f'rows total:    {len(df)}')
print(f'base reads:    {df["is_base"].sum()}')
print(f'science reads: {(~df["is_base"]).sum()}')
print()
for d in sorted(df['date'].unique()):
    day = df[df['date'] == d]
    stations = sorted(day.loc[~day['is_base'], 'point_along_profile_m'])
    print(f'{d}: {day["is_base"].sum()} base reads, science stations {stations}')

`df.describe()` summarises every numeric column of `df` in one table: count, [mean](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#mean), [standard deviation](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#standard-deviation-s), minimum, quartiles and maximum. Question 1.1 reads the `gravity_mgal` and `elev_rel_base_m` columns of this table, and Question 2.3 reads the `sd_mgal` column.

In [ ]:
df.describe()

**Question 1.1.** The `describe()` table gives the minimum and maximum of every column. Two of them matter here: `gravity_mgal` and `elev_rel_base_m`.

1. Report the range of `gravity_mgal` across the whole survey (max minus min), in mGal.
2. The [free-air gradient](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#free-air-gradient) you measured and derived in HW1 is about 0.3086 mGal per metre. Multiply it by the range of `elev_rel_base_m` to estimate how much spread in `gravity_mgal` station elevation alone produces.
3. Compare the two numbers. Your elevation estimate may come out larger than the range you are comparing it against; if it does, that is a result rather than an arithmetic slip, and part of your answer is saying what it implies. What does the comparison say about reading the raw column for a sinkhole whose signal is a few hundredths of a milligal?

*(Your answer):*

## Part 2: Drift, and tying three field days together

In HW1 the drift line went through two ground reads, and the closure came out at zero by construction. Here each day has six to eleven base reads, so the drift is a [least-squares](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#least-squares) line with scatter about it, and the fit returns an uncertainty on the slope from its [covariance matrix](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#covariance-matrix). HW1 Part 4 explains what the fit minimises and what sets the slope's [sigma](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#sigma); the same `numpy.polyfit(..., cov=True)` call does the work here.

Two corrections follow, each applied one field day at a time.

1. **Drift correction, per day.** Fit a straight line through the day's base reads against time and subtract `slope × time` from every reading on that day. Time is measured from the day's first reading.
2. **Base tie, per day.** Across days, the level the meter reads at can shift. To put all three days on a common reference we subtract each day's mean drift-corrected base reading from every reading on that day. Every day's base then sits at 0 mGal, and every science station's number is a relative gravity in mGal against the base. A gravimeter is a [relative instrument](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#relative-and-absolute-measurement): the survey needs differences between stations and never needs the absolute value at the base.

After both steps the base reads should scatter about zero. Their standard deviation is the survey's repeatability: how closely the same place reads the same number once the corrections are in. The cell pools that scatter across the three days; the pooled value is the noise floor HW2b uses to judge whether an anomaly is real.

This cell selects the base reads from `df` with the `is_base` column and plots `gravity_mgal` against clock time `time_est` with `px.scatter`, one color per field day. The figure shows the raw base reads before any correction. The drift fit two cells below works on these same reads.

In [ ]:
base = df[df['is_base']].copy()
fig = px.scatter(
    base,
    x='time_est',
    y='gravity_mgal',
    color=base['date'].astype(str),
    title='Base-station reads before correction (color = field day)',
    labels={'time_est': 'Time', 'gravity_mgal': 'Gravity reading (mGal)', 'color': 'Date'},
)
fig.update_traces(marker=dict(size=10))
fig.show()

**Figure description:** A scatter plot of the 23 base-station reads before any correction, clock time on the x-axis and gravity reading (mGal) on the y-axis, colored by field day. The three days appear as three separated clusters because of the gaps between sessions. Within a day the reads trend or scatter; across days they sit at different levels. Non-visual path: the next cell prints, for each day, the number of base reads, the fitted drift slope with its uncertainty, the base reference level, and the scatter of the corrected base reads, which carry the same information as the plot.

This cell carries out the two steps described above, one field day at a time, and writes the result to a new column `relative_gravity` on `df`. For each day:

- It reads that day's rows and, within them, the base reads. Time comes from `time_since_beg_min` with the day's first reading subtracted, so each day starts at 0 minutes.
- `np.polyfit(..., 1, cov=True)` fits a straight line of `gravity_mgal` against time through the base reads by least squares. The slope is the drift rate in mGal per minute; the square root of the `[0, 0]` entry of the covariance matrix is its 1-sigma uncertainty.
- It subtracts `slope × time` from every reading on the day, base and science alike.
- It takes the mean of the drift-corrected base reads as the day's base reference and subtracts it from every reading on the day. Those values go into `relative_gravity`.
- It prints the day's number of base reads and span in minutes, the slope with its uncertainty, the base reference, and the standard deviation and [standard error](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#standard-error-se) of the corrected base reads.

After the loop it pools the corrected base reads from all three days into one standard deviation, `pooled_sd`, with one degree of freedom removed per day for the mean that was subtracted, and prints it. HW2b's Parts 7 and 8 read `pooled_sd`.

In [ ]:
df['relative_gravity'] = np.nan
corrected_base = []   # drift-corrected, base-tied base reads from every day

for d in sorted(df['date'].unique()):
    day_mask = df['date'] == d
    day = df[day_mask]
    day_base = day[day['is_base']]

    # Day-local time: 0 at the day's first reading.
    t0 = day['time_since_beg_min'].min()
    day_t = day['time_since_beg_min'] - t0
    base_t = day_base['time_since_beg_min'] - t0

    # 1. Fit the drift through the day's base reads. cov=True returns the covariance
    #    matrix; the square root of its [0, 0] entry is the 1-sigma uncertainty on
    #    the slope.
    (slope, intercept), cov = np.polyfit(base_t, day_base['gravity_mgal'], 1, cov=True)
    slope_sigma = np.sqrt(cov[0, 0])

    # 2. Drift-correct every reading on this day.
    drift_corrected = day['gravity_mgal'] - slope * day_t

    # 3. Base tie: subtract the day's mean drift-corrected base reading.
    base_reference = drift_corrected[day['is_base']].mean()
    df.loc[day_mask, 'relative_gravity'] = drift_corrected - base_reference

    base_after = drift_corrected[day['is_base']] - base_reference
    corrected_base.append(base_after.to_numpy())

    print(f'{d}: N_base = {len(day_base):2d}, span = {day_t.max():3.0f} min')
    print(f'    drift slope     = {slope:+.6f} +/- {slope_sigma:.6f} mGal/min (1 sigma)')
    print(f'    base reference  = {base_reference:.4f} mGal')
    print(f'    corrected base reads: SD = {base_after.std(ddof=1):.4f} mGal, '
          f'SE of the day mean = {base_after.std(ddof=1) / np.sqrt(len(day_base)):.4f} mGal')

# Pooled repeatability: one standard deviation over all 23 corrected base reads,
# with one degree of freedom removed for each day's mean.
all_base = np.concatenate(corrected_base)
n_days = len(corrected_base)
pooled_sd = np.sqrt(np.sum(all_base**2) / (len(all_base) - n_days))
print()
print(f'pooled SD of the {len(all_base)} drift-corrected base reads: {pooled_sd:.4f} mGal')
print('(the survey noise floor used in HW2b)')

This cell selects the base reads again and plots `relative_gravity` against clock time, colored by field day, with the same `px.scatter` call as the plot before correction. The points are the same 23 base reads after the drift correction and the base tie.

In [ ]:
base_after = df[df['is_base']]
fig = px.scatter(
    base_after,
    x='time_est',
    y='relative_gravity',
    color=base_after['date'].astype(str),
    title='Base-station reads after drift correction and base tie',
    labels={'time_est': 'Time', 'relative_gravity': 'Relative gravity (mGal, base = 0)', 'color': 'Date'},
)
fig.update_traces(marker=dict(size=10))
fig.show()

**Figure description:** The same 23 base-station reads after the drift correction and base tie, clock time on the x-axis and relative gravity (mGal) on the y-axis, colored by field day. Every day now scatters about the horizontal line at 0 mGal. The width of that scatter is the repeatability printed by the previous cell. Non-visual path: the previous cell prints each day's corrected-base standard deviation and the pooled value.

**Question 2.1.** The printout gives three drift slopes, each with a 1-sigma uncertainty.

1. HW1 used the normal rule: a fitted value lands within 2 sigma of the truth about 95 per cent of the time, so a magnitude larger than twice its uncertainty is one the scatter is unlikely to have produced. By that bar, which of the three slopes is resolved? Which are consistent with zero?
2. For the resolved day, what does the sign of the slope say about how the reading changed while the operator worked?
3. Multiply that slope by the day's span in minutes to get the drift accumulated between the day's first and last reads. Compare it with two things: that day's corrected-base SD, and the size of a sinkhole anomaly, which for a cavity like this one is a few hundredths of a mGal. Would the day's stations have been usable without the drift correction?

*(Your answer):*

**Question 2.2.** The base reference on April 18 is about 0.1 mGal below the references on April 4 and April 6, two and twelve days earlier. HW1 Question 4.4 asked you to forecast what a return visit would read. The three sessions ran at different times of day: April 4 from 18:29, April 6 from 10:27, April 18 from 15:52.

1. Name two different causes that could put the same place 0.1 mGal lower on a later day. One is the instrument. For the other, look at the session times and at the catalyst list from Week 3. Which of the two, if either, can this survey rule out?
2. The base tie removes that offset whatever caused it. What does the tie assume about the offset within a single day, and what kind of event during a day would break that assumption?

*(Your answer):*

**Question 2.3.** Two different numbers describe the uncertainty of a reading in this survey.

1. From the `describe()` table in Part 1, report the median of `sd_mgal` and its range. This is the instrument's own uncertainty on a single reading.
2. The corrected base reads on each day scatter by the printed SD. Compare it with the typical `sd_mgal`. Why is the same place read repeatedly more scattered than the instrument says one reading should be? Name one source of scatter that a per-reading instrument uncertainty cannot include.
3. When Part 7 asks whether a low on the profile is real, which of the two numbers is the right yardstick, and why?

*(Your answer):*

## Part 3: Free air and the slab at the textbook density

**Free-air correction.** Gravity falls by about 0.3086 mGal for every metre of elevation, the [free-air gradient](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#free-air-gradient) you measured in the stairwell and derived from `g = GM/R²` in HW1. The stations here sit within 2.4 m of the base elevation, so the correction adds back what a station above the base lost, and subtracts from a station below it:

$$g_\text{FA} = g_\text{relative} + 0.3086\,h$$

where `h` is `elev_rel_base_m`, positive above the base.

**Bouguer slab correction.** The free-air correction treats the space between a station and base level as empty. A station above the base has rock between it and base level, and that rock pulls on the meter. Burger §6.3.4 models it as an infinite flat slab of thickness `h` and density ρ, whose attraction is `2πGρh`, and removes it:

$$g_\text{B} = g_\text{FA} - 2\pi G \rho\, h$$

Week 3 wrote the coefficient at the board as `0.04193 ρ h` mGal, with ρ in g/cm³ and `h` in metres. The cell below computes the same coefficient in SI units (ρ in kg/m³, the result converted to mGal) and prints it, so the two forms can be checked against each other. The density used here is 2.67 g/cm³, the value most texts quote for the average continental crust. Week 3 worked Burger's problem at 2.50 g/cm³.

The reduced profile that comes out is the **Bouguer anomaly**: what remains of the gravity differences between stations after the instrument and the station elevations have been accounted for. The cell prints the table for every science station and the sizes of the largest terms, then plots the anomaly along the line.

This cell applies the two elevation corrections above at the textbook density and produces the Part 3 table.

- It defines the constants `G` and `FAC_GRADIENT` (0.3086 mGal per metre) and a function `bouguer_coefficient(rho_kg_m3)` that returns `2πGρ` in mGal per metre for a density in kg/m³. Later parts call this function at other densities.
- It evaluates the coefficient at `RHO_TEXTBOOK` (2670 kg/m³, which is 2.67 g/cm³), stores it as `coeff_textbook`, and prints it beside the board form `0.04193 × 2.67` for comparison.
- It adds two columns to `df`: `fac_corrected`, which is `relative_gravity` plus `FAC_GRADIENT` times `elev_rel_base_m`, and `bouguer_2670`, which is `fac_corrected` minus `coeff_textbook` times `elev_rel_base_m`.
- It builds `science`, the twenty science stations sorted by `point_along_profile_m`, and prints a table of their position, date, elevation, relative gravity, free-air-corrected gravity and Bouguer anomaly, rounded to four decimals.
- It prints the largest free-air term and the largest slab term on the line, each as an absolute value in mGal.

Later cells read `science`, `coeff_textbook` and `bouguer_coefficient`.

In [ ]:
G = 6.674e-11                # m^3 / (kg s^2)
FAC_GRADIENT = 0.3086        # mGal per metre

def bouguer_coefficient(rho_kg_m3):
    """Slab attraction per metre of thickness, in mGal per metre."""
    return 2 * np.pi * G * rho_kg_m3 * 1e5    # 1 mGal = 1e-5 m/s^2

RHO_TEXTBOOK = 2670          # kg/m^3, which is 2.67 g/cm^3
coeff_textbook = bouguer_coefficient(RHO_TEXTBOOK)
print(f'Bouguer coefficient at 2.67 g/cm^3: {coeff_textbook:.4f} mGal per metre (SI route)')
print(f'board form 0.04193 x 2.67:          {0.04193 * 2.67:.4f} mGal per metre')

df['fac_corrected'] = df['relative_gravity'] + FAC_GRADIENT * df['elev_rel_base_m']
df['bouguer_2670'] = df['fac_corrected'] - coeff_textbook * df['elev_rel_base_m']

science = df[~df['is_base']].sort_values('point_along_profile_m').reset_index(drop=True)
table = science[['point_along_profile_m', 'date', 'elev_rel_base_m',
                 'relative_gravity', 'fac_corrected', 'bouguer_2670']]
print()
print(table.round(4).to_string(index=False))

h = science['elev_rel_base_m']
print()
print(f'largest free-air term on the line:   {(FAC_GRADIENT * h).abs().max():.4f} mGal')
print(f'largest slab term at 2.67 g/cm^3:    {(coeff_textbook * h).abs().max():.4f} mGal')

This cell plots `bouguer_2670` from `science` against `point_along_profile_m` with `px.scatter`, colored by field day. Each marker is one science station. The values are the same ones in the `bouguer_2670` column of the table above.

In [ ]:
fig = px.scatter(
    science,
    x='point_along_profile_m',
    y='bouguer_2670',
    color=science['date'].astype(str),
    title='Bouguer anomaly along the profile, slab density 2.67 g/cm³',
    labels={'point_along_profile_m': 'Along-profile distance (m); base at 100 m',
            'bouguer_2670': 'Bouguer anomaly (mGal)', 'color': 'Date'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** A scatter plot of the Bouguer anomaly (mGal, y-axis) against along-profile distance (metres, x-axis) for the twenty science stations, colored by field day, with the base at 100 m. The anomaly varies along the line by about a tenth of a milligal, and the questions below ask you to find its lowest station from the table. Non-visual path: the previous cell prints the along-profile distance, elevation and anomaly for every station.

**Question 3.1.** A sinkhole is a cavity in the limestone, filled with air, water or loose sediment in place of rock, so its [density contrast](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#property-contrast) with the surrounding rock is negative.

1. Before reading the table: does a body with less mass than its surroundings raise or lower gravity at the surface above it? So which sign of anomaly marks a sinkhole?
2. Now read the table. Which station has the lowest Bouguer anomaly, and what is its value? Name the stations either side of it and their values. How many stations carry the low, and does it look like an isolated feature or part of a longer fall? Describe the shape of the profile from 130 m to the end of the line.

*(Your answer):*

**Question 3.2.** Station 180 sits 2.376 m above the base, the highest point on the line, and station 170 sits 1.555 m above the base. The table gives both stations' relative gravity before either elevation correction, and their anomaly after both.

1. Work the two correction terms for station 180 by hand, with signs: the free-air term at 0.3086 mGal/m and the slab term at 0.04193 × 2.67 mGal/m. State what each sign means.
2. Add both terms to station 180's relative gravity from the table. Do you recover the Bouguer anomaly the table prints for that station?
3. From the table, how much did the relative gravity change between stations 170 and 180 before the corrections, and how much of that difference remains in the Bouguer anomaly? What fraction of the raw step between the two stations was elevation?

*(Your answer):*

## Part 4: The regional and the residual

The Bouguer anomaly in Part 3 varies along the line for more than one reason. A sinkhole a few metres across produces a change over a few tens of metres. Structure deeper or broader than the target produces a change that runs the whole 200 m and beyond: a dipping contact under the field, a thickening of the sediment cover across the campus, or the change in normal gravity with latitude that Week 4 placed in the correction sequence. The long-wavelength part is the **regional**. What is left after it is removed is the [[**residual**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual)](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual), and the residual is what HW2b interprets.

Week 4 wrote the reduction at the board as `drift → latitude → free air → slab → residual`. Parts 2 and 3 covered drift, free air and the slab. This part covers the residual. The published exercise this survey comes from fits a local straight-line trend to the reduced profile and interprets what remains, and Weeks 7 and 8 build the same regional and residual separation into the magnetics corrections.

A straight line across 200 m is the simplest model of a regional. The cell below fits one by [least squares](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#least-squares) to the Bouguer anomaly at 2.67 g/cm³ against along-profile distance, with a 1-sigma uncertainty on the slope from the [covariance matrix](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#covariance-matrix) as in Part 2, and subtracts it from every station. It prints the slope, the change the line implies from one end of the profile to the other, the trend of station elevation itself along the line, and the anomaly, regional and residual at every station. The residual has zero mean by construction. For comparison the cell also fits and removes a second-order polynomial and prints where each of the two residuals has its minimum.

Fitting a line is a choice. A polynomial of higher order, or a line fitted only to the stations far from the target, gives a different residual from the same twenty readings, and nothing in the readings fixes the order of the fit. The reduction density in Part 3 was the first choice of this kind in the notebook; the regional is the second. The plot shows the anomaly, the fitted regional and the residual on one set of axes.

This cell fits and removes the straight-line regional described above.

- It reads `point_along_profile_m` and `bouguer_2670` from `science` into the arrays `x` and `anomaly_2670`.
- `np.polyfit(x, anomaly_2670, 1, cov=True)` fits a straight line by least squares. The slope `trend_slope` is in mGal per metre along the line and `trend_slope_sigma` is its 1-sigma uncertainty from the covariance matrix.
- It adds two columns to `df`: `regional_2670`, the fitted line evaluated at every station, and `residual_2670`, which is `bouguer_2670` minus `regional_2670`. It then rebuilds `science` so the new columns are available there.
- A second `np.polyfit` call, of `elev_rel_base_m` against `x`, gives `elev_trend`, the slope of station elevation along the line in metres per metre.
- It prints the regional slope with its uncertainty, the change in the regional across the line (slope times line length), the elevation trend, and a table of position, elevation, anomaly, regional and residual for every station.
- For comparison it fits a second-order polynomial with `np.polyfit(..., 2)`, evaluates it with `np.polyval`, subtracts it to form `residual_quadratic`, and prints the station and value of the minimum of each of the two residuals.

HW2b's Parts 5 to 8 read `residual_2670`.

In [ ]:
x = science['point_along_profile_m'].to_numpy()
anomaly_2670 = science['bouguer_2670'].to_numpy()

# Straight-line regional by least squares against along-profile distance. cov=True
# gives the covariance matrix; the square root of its [0, 0] entry is the 1-sigma
# uncertainty on the slope, as in the drift fit of Part 2.
(trend_slope, trend_intercept), trend_cov = np.polyfit(x, anomaly_2670, 1, cov=True)
trend_slope_sigma = np.sqrt(trend_cov[0, 0])

df['regional_2670'] = trend_slope * df['point_along_profile_m'] + trend_intercept
df['residual_2670'] = df['bouguer_2670'] - df['regional_2670']
science = df[~df['is_base']].sort_values('point_along_profile_m').reset_index(drop=True)

line_length = x.max() - x.min()                                   # metres
elev_trend = np.polyfit(x, science['elev_rel_base_m'], 1)[0]       # m of elevation per m along the line

print(f'regional slope:                  {trend_slope:+.6f} +/- {trend_slope_sigma:.6f} mGal per metre (1 sigma)')
print(f'change across the {line_length:.0f} m line:    {trend_slope * line_length:+.4f} mGal')
print(f'trend of station elevation:      {elev_trend:+.5f} m of elevation per metre along the line')
print()
print(science[['point_along_profile_m', 'elev_rel_base_m', 'bouguer_2670', 'regional_2670', 'residual_2670']]
      .round(4).to_string(index=False))

# The same step with a second-order polynomial as the regional, for comparison.
residual_quadratic = anomaly_2670 - np.polyval(np.polyfit(x, anomaly_2670, 2), x)
print()
for label, r in [('straight line', science['residual_2670'].to_numpy()),
                 ('second-order polynomial', residual_quadratic)]:
    i = int(np.argmin(r))
    print(f'regional = {label:24s} residual minimum at {x[i]:.0f} m, value {r[i]:+.4f} mGal')


This cell draws three series on one set of axes with `px.scatter` and two `fig.add_scatter` calls: `bouguer_2670` as circles, `regional_2670` as a dashed line whose slope appears in the legend, and `residual_2670` as diamonds. `fig.add_hline` draws a dotted line at zero, the level the residual is measured from.

In [ ]:
fig = px.scatter(
    science,
    x='point_along_profile_m',
    y='bouguer_2670',
    title='Bouguer anomaly at 2.67 g/cm³, its straight-line regional, and the residual',
    labels={'point_along_profile_m': 'Along-profile distance (m); base at 100 m',
            'bouguer_2670': 'Gravity (mGal)'},
)
fig.update_traces(marker=dict(size=12), name='Bouguer anomaly', showlegend=True)
fig.add_scatter(
    x=x, y=science['regional_2670'], mode='lines',
    name=f'regional: {trend_slope:+.5f} mGal/m',
    line=dict(color=px.colors.qualitative.Safe[3], width=3, dash='dash'),
)
fig.add_scatter(
    x=x, y=science['residual_2670'], mode='markers', name='residual',
    marker=dict(size=12, symbol='diamond', color=px.colors.qualitative.Safe[1]),
)
fig.add_hline(y=0, line_dash='dot')
fig.show()


**Figure description:** A plot with along-profile distance (metres, x-axis) and gravity (mGal, y-axis) carrying three series: the Bouguer anomaly at 2.67 g/cm³ as circular markers, the fitted straight-line regional as a dashed line whose slope is given in the legend, and the residual as diamond markers scattered about a dotted horizontal line at zero. Non-visual path: the previous cell prints the slope with its uncertainty and the anomaly, regional and residual at every station.

**Question 4.1.** The printout gives the slope of the fitted regional with its 1-sigma uncertainty, and the table gives the anomaly, the regional and the residual at every station.

1. Is the slope resolved, in the sense of Question 2.1? Report the change in the regional from one end of the line to the other and compare it with the value of the lowest Bouguer anomaly you reported in Question 3.1.
2. Which station holds the lowest residual, and what is its value? Compare its position and value with the minimum you found in Question 3.1.
3. What could produce a straight-line change in gravity of this size along a 200 m line? Name two candidates. For each, say what would have to be true of the site or the survey for it to produce a trend this large. HW2b's Question 8.3 returns to one of them.
4. A third candidate is in the printout, which also gives the trend of station elevation along the line, in metres of elevation per metre of distance. Report it. Along a line whose ground rises steadily, an error in the slab density produces a straight-line trend in the Bouguer anomaly, and a fitted regional absorbs it. If the density used in Part 3 were too high, which way would that trend run here? Compare its sign with the sign of the regional you fitted.

*(Your answer):*

**Question 4.2.** The last two lines of the printout compare the residual left by a straight line with the residual left by a second-order polynomial.

1. Report the minimum left by each of the two regionals. What fraction of the low did the second-order fit remove? Then say what stayed the same between them.
2. Both curves are models of the regional. What in the data, or outside it, could justify choosing one over the other?
3. In one sentence, what should a survey report state about its regional removal?

*(Your answer):*

## How to submit

1. Run all cells from the top. (`Runtime → Run all` in Colab.)
2. Make sure all your short answers are filled in.
3. `File → Download → Download .ipynb`.
4. Rename the file to `HW2a_LASTNAME.ipynb` (e.g., `HW2a_Smith.ipynb`).
5. Upload to the Brightspace HW2a dropbox by **Sunday October 4, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.